# Teoria de Feature Engineering

_Convertir datos crudos en features que un modelo pueda aprovechar: seleccion, imputacion, encoding, transformaciones, escalado, PCA y embeddings._

**Modulo 1 — Feature Engineering & Feature Stores** | DSRP Machine Learning Engineering
**Profesor:** Miguel Arquez

![Feature Engineering](assets/header.png)


## Introduccion

El **feature engineering** es el arte de convertir datos crudos en entradas
numericas (las "features") con las que un modelo de machine learning puede
aprender bien. Una buena analogia: el modelo es un cocinero y las features son
los ingredientes ya picados y medidos. Por muy buen cocinero que sea, si le das
ingredientes en mal estado el plato no saldra bien.

Un modelo solo es tan bueno como la **representacion** que recibe. Las
transformaciones correctas logran tres cosas:

- hacen que los patrones sean *separables* (lineal o localmente),
- estabilizan la optimizacion (que el entrenamiento converja),
- y codifican conocimiento del dominio que el modelo, de otro modo, tendria que
  inferir desde cero.

En este notebook recorremos el flujo tipico de feature engineering. Para cada
tecnica vemos primero **(a)** la intuicion y luego **(b)** un ejemplo ejecutable
sobre el dataset real del **Titanic**:

1. **Seleccion de variables** (filter, wrapper, embedded) — *que columnas conservar*
2. **Imputacion** de valores faltantes
3. **Encoding** de categoricas (label / ordinal, one-hot, feature hashing)
4. **Transformaciones** (log, Box-Cox / Yeo-Johnson, binning)
5. **Escalado** (z-score, robusto, min-max, L2)
6. **Reduccion de dimensionalidad** (PCA)
7. **Embeddings** (vectores densos aprendidos)

Cerramos con una **tabla resumen** de cuando usar cada tecnica.

> Una distincion util desde el principio: la **seleccion** de variables *elige un
> subconjunto de las columnas originales* (las mantiene interpretables), mientras
> que la **extraccion** de variables (como PCA) *crea nuevas columnas* combinando
> las originales. Ambas reducen dimensionalidad, pero solo la seleccion conserva
> el significado de cada feature.


### El pipeline de feature engineering de un vistazo

Antes de entrar en cada tecnica, este es el flujo general que recorren los datos:
de **datos crudos** a una **matriz de features** lista para el modelo. El diagrama
de abajo se dibuja con matplotlib (se genera al ejecutar la celda).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(12, 2.6))
ax.set_xlim(0, 12); ax.set_ylim(0, 3); ax.axis("off")

etapas = [
    ("Datos\ncrudos", "#cfe8ff"),
    ("Imputacion +\nseleccion", "#cfe8ff"),
    ("Transformaciones\n(encoding, scaling,\nbinning, PCA...)", "#ffe2b3"),
    ("Matriz de\nfeatures", "#d6f5d6"),
    ("Modelo de\nML", "#f3d1ff"),
]
w, h, gap = 2.0, 1.4, 0.35
x = 0.25
centros = []
for texto, color in etapas:
    box = FancyBboxPatch((x, 0.8), w, h, boxstyle="round,pad=0.05",
                         fc=color, ec="#444", lw=1.3)
    ax.add_patch(box)
    ax.text(x + w / 2, 0.8 + h / 2, texto, ha="center", va="center", fontsize=10)
    centros.append(x + w / 2)
    x += w + gap

for i in range(len(centros) - 1):
    ax.add_patch(FancyArrowPatch((centros[i] + w / 2, 1.5),
                                 (centros[i + 1] - w / 2, 1.5),
                                 arrowstyle="-|>", mutation_scale=18, color="#333", lw=1.5))

ax.text(6, 2.7, "Pipeline de feature engineering", ha="center",
        fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

## Setup y datos

Usamos el dataset del Titanic que viene con seaborn. Si estas sin conexion,
`seaborn.load_dataset` no podra descargarlo, asi que lo envolvemos en un
`try/except` y caemos a un pequeno dataframe sintetico con las mismas columnas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(0)

def load_titanic():
    try:
        import seaborn as sns
        df = sns.load_dataset("titanic")
        if df is None or len(df) == 0:
            raise ValueError("vacio")
        print("Dataset real del Titanic cargado desde seaborn.")
        return df
    except Exception as e:  # sin conexion / sin seaborn-data
        print(f"Usando datos sinteticos ({type(e).__name__}: {e})")
        n = 500
        df = pd.DataFrame({
            "survived": np.random.randint(0, 2, n),
            "pclass": np.random.choice([1, 2, 3], n, p=[0.2, 0.2, 0.6]),
            "sex": np.random.choice(["male", "female"], n, p=[0.65, 0.35]),
            "age": np.clip(np.random.normal(29, 14, n), 0.4, 80).round(1),
            "sibsp": np.random.poisson(0.5, n),
            "parch": np.random.poisson(0.4, n),
            "fare": np.round(np.random.exponential(32, n), 4),
            "embarked": np.random.choice(["S", "C", "Q"], n, p=[0.72, 0.19, 0.09]),
            "class": np.random.choice(["First", "Second", "Third"], n),
            "embark_town": np.random.choice(
                ["Southampton", "Cherbourg", "Queenstown"], n, p=[0.72, 0.19, 0.09]),
        })
        return df

df = load_titanic()
print(df.shape)
df.head()

In [ ]:
# Un vistazo rapido a tipos y valores faltantes: las decisiones de ingenieria
# dependen de esto.
df.info()
df.isna().sum()

## 1. Seleccion de variables

**Intuicion primero.** No todas las columnas ayudan. Algunas son ruido, otras son
redundantes (muy correlacionadas entre si) y otras simplemente no tienen relacion
con el objetivo. Quedarte con menos variables —pero mejores— suele dar modelos
**mas simples, mas rapidos, menos propensos al sobreajuste y mas faciles de
interpretar**. La seleccion *elige un subconjunto de las columnas originales*; no
las transforma (eso lo hace PCA, en la seccion 6).

Hay tres familias clasicas:

- **Filter (filtros).** Puntuan cada variable con un estadistico *independiente del
  modelo* y se quedan con las mejores. Rapidos y baratos. Ejemplos: varianza casi
  nula (`VarianceThreshold`), correlacion con el objetivo, **informacion mutua**
  (`mutual_info_classif`, captura relaciones no lineales) y **chi-cuadrado**
  (`chi2`, para features no negativas vs un objetivo categorico).
- **Wrapper (envolventes).** Usan *el propio modelo* para evaluar subconjuntos de
  variables, entrenando muchas veces. Mas precisos pero caros. Ejemplo: **RFE**
  (eliminacion recursiva: entrena, descarta la peor variable, repite).
- **Embedded (embebidos).** La seleccion ocurre *dentro* del entrenamiento del
  modelo. Ejemplos: **L1 / Lasso** (lleva coeficientes a 0 exacto) e **importancia
  de variables de los arboles** (`feature_importances_`).

**Cuando usar cada una.** Empieza por filtros para una criba barata; usa embedded
(L1, arboles) cuando ya entrenas ese tipo de modelo —sale casi gratis—; reserva
wrappers (RFE) para cuando puedes pagar el coste y quieres exprimir el subconjunto
optimo.

In [ ]:
from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_classif, chi2, SelectKBest, RFE)
from sklearn.preprocessing import MinMaxScaler

# Construimos una matriz numerica/codificada simple a partir del Titanic para
# ilustrar la seleccion. Imputamos rapido (mediana / moda) solo para este ejemplo.
sel = pd.DataFrame()
sel["age"] = df["age"].fillna(df["age"].median())
sel["fare"] = df["fare"].fillna(df["fare"].median())
sel["sibsp"] = df["sibsp"].fillna(0)
sel["parch"] = df["parch"].fillna(0)
sel["pclass"] = df["pclass"].fillna(3)
sel["sex_male"] = (df["sex"] == "male").astype(int)          # label/binario
sel["family_size"] = sel["sibsp"] + sel["parch"] + 1
y_sel = df["survived"].astype(int)
print("Features candidatas:", list(sel.columns))
sel.head()

In [ ]:
# (a) FILTER — varianza casi nula: descarta columnas casi constantes.
vt = VarianceThreshold(threshold=0.0)
vt.fit(sel)
print("Varianza por columna:\n", sel.var().round(3).to_dict())

# (b) FILTER — correlacion con el objetivo (lineal, |r|).
corr = sel.apply(lambda c: np.corrcoef(c, y_sel)[0, 1]).abs().sort_values(ascending=False)
print("\n|correlacion| con survived:\n", corr.round(3).to_string())

# (c) FILTER — informacion mutua (captura dependencias NO lineales).
mi = pd.Series(mutual_info_classif(sel, y_sel, random_state=0),
               index=sel.columns).sort_values(ascending=False)
print("\nInformacion mutua con survived:\n", mi.round(3).to_string())

In [ ]:
# (d) FILTER — chi-cuadrado: requiere features NO negativas. Escalamos a [0,1].
sel_pos = MinMaxScaler().fit_transform(sel)
chi_stats, chi_p = chi2(sel_pos, y_sel)
chi_tbl = pd.DataFrame({"chi2": chi_stats, "p_value": chi_p},
                       index=sel.columns).sort_values("chi2", ascending=False)
print("Chi-cuadrado (mayor chi2 / menor p = mas dependiente del objetivo):")
print(chi_tbl.round(4))

# SelectKBest deja las k mejores segun el puntaje elegido (aqui, informacion mutua).
kbest = SelectKBest(score_func=mutual_info_classif, k=4).fit(sel, y_sel)
print("\nTop-4 por informacion mutua:", list(sel.columns[kbest.get_support()]))

In [ ]:
# (e) WRAPPER — RFE: entrena un modelo, elimina la peor variable y repite.
from sklearn.linear_model import LogisticRegression

rfe = RFE(LogisticRegression(max_iter=1000), n_features_to_select=4)
rfe.fit(MinMaxScaler().fit_transform(sel), y_sel)
print("RFE — seleccionadas:", list(sel.columns[rfe.support_]))
print("RFE — ranking (1 = elegida):", dict(zip(sel.columns, rfe.ranking_)))

In [ ]:
# (f) EMBEDDED — L1/Lasso (coeficientes a 0) e importancia de arboles.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# L1: la regularizacion lleva a cero los coeficientes de las variables irrelevantes.
l1 = LogisticRegression(penalty="l1", solver="liblinear", C=0.5, max_iter=1000)
l1.fit(MinMaxScaler().fit_transform(sel), y_sel)
l1_coef = pd.Series(l1.coef_.ravel(), index=sel.columns)
print("Coeficientes L1 (los =0 quedan descartados):")
print(l1_coef.round(3).to_string())

# Arboles: importancia por reduccion de impureza.
rf = RandomForestClassifier(n_estimators=200, random_state=0).fit(sel, y_sel)
imp = pd.Series(rf.feature_importances_, index=sel.columns).sort_values(ascending=False)
print("\nImportancia de variables (Random Forest):")
print(imp.round(3).to_string())

## 2. Imputacion de valores faltantes

**Intuicion primero.** Casi ningun modelo de sklearn acepta `NaN`: si dejas
huecos, fallara o tendras que tirar filas/columnas enteras y perder informacion.
**Imputar** es rellenar esos huecos con un valor razonable. Pero ojo: la forma de
rellenar *codifica un supuesto*, y un mal supuesto sesga al modelo. El Titanic
tiene faltantes reales en `age` y `embark_town`, asi que trabajamos con datos de
verdad.

Opciones segun el tipo de variable y el mecanismo de los faltantes:

- **`SimpleImputer` (media / mediana / moda).** Rapido. **media** para numericas
  simetricas, **mediana** para numericas sesgadas o con outliers (mas robusta),
  **most_frequent** (moda) para categoricas. Reduce la varianza y puede distorsionar
  relaciones, pero es un baseline solido.
- **`KNNImputer`.** Rellena cada hueco con el promedio de las $k$ filas mas
  parecidas (vecinos). Capta relaciones entre columnas; mas costoso y sensible a la
  escala (conviene escalar antes).
- **Variable indicadora de faltante (`add_indicator=True`).** Agrega una columna
  binaria "este valor estaba ausente". Util cuando *el hecho de faltar es
  informativo* (p. ej. no declarar la edad puede correlacionar con la clase del
  pasaje). El modelo puede aprender de la ausencia misma.

**Regla practica:** ajusta el imputador **solo en train** (la mediana se calcula en
train y se reutiliza en test) para no filtrar informacion.

In [ ]:
from sklearn.impute import SimpleImputer, KNNImputer

print("Faltantes antes de imputar:")
print(df[["age", "fare", "embark_town"]].isna().sum())

# (a) SimpleImputer — mediana para 'age' (sesgada), moda para 'embark_town'.
age_median = SimpleImputer(strategy="median")
age_imp = age_median.fit_transform(df[["age"]])
print(f"\n'age': mediana aprendida = {age_median.statistics_[0]:.1f} "
      f"| NaN restantes = {np.isnan(age_imp).sum()}")

town_mode = SimpleImputer(strategy="most_frequent")
town_imp = town_mode.fit_transform(df[["embark_town"]])
print(f"'embark_town': moda = {town_mode.statistics_[0]!r} "
      f"| NaN restantes = {pd.isna(town_imp).sum()}")

In [ ]:
# (b) Variable indicadora: marca DONDE faltaba antes de rellenar.
imp_ind = SimpleImputer(strategy="median", add_indicator=True)
age_with_flag = imp_ind.fit_transform(df[["age"]])
print("Salida con add_indicator -> 2 columnas: [valor imputado, 1=faltaba]")
print(pd.DataFrame(age_with_flag[:8], columns=["age_imp", "age_was_missing"]))
print("Total de edades que estaban ausentes:", int(age_with_flag[:, 1].sum()))

In [ ]:
# (c) KNNImputer — usa columnas correlacionadas para estimar el hueco.
# Lo aplicamos a un bloque numerico; escalar primero ayuda a la distancia.
from sklearn.preprocessing import StandardScaler

knn_cols = ["age", "fare", "sibsp", "parch", "pclass"]
block = df[knn_cols].copy()
scaler_knn = StandardScaler()
block_scaled = scaler_knn.fit_transform(block)        # NaN se preservan
knn = KNNImputer(n_neighbors=5)
block_knn = knn.fit_transform(block_scaled)
# Volvemos a la escala original solo para inspeccionar 'age'.
block_back = scaler_knn.inverse_transform(block_knn)
print("Faltantes tras KNNImputer:", int(np.isnan(block_knn).sum()))
print("Ejemplo: edades imputadas por KNN (primeras que estaban NaN):")
nan_rows = df["age"].isna().to_numpy().nonzero()[0][:5]
print(np.round(block_back[nan_rows, 0], 1))

## 3. Encoding de categoricas

Los modelos necesitan numeros, no texto. **Codificar** una variable categorica es
mapear sus niveles a numeros, y hay varias formas con implicaciones muy distintas.
Veremos tres: **label / ordinal encoding**, **one-hot** y **feature hashing**.

### 3a. Label encoding y ordinal encoding

**Intuicion primero.** Lo mas directo es asignar un entero a cada categoria:
`S -> 0`, `C -> 1`, `Q -> 2`. Eso es **label encoding** (para el objetivo) u
**ordinal encoding** (para las features). El problema: introduce un **orden y una
distancia falsos**. Un modelo lineal o de distancia interpretara que `Q (2)` es
"mayor" que `S (0)` y que esta "al doble de distancia" que `C (1)`, lo cual no
tiene sentido entre puertos.

**Cuando SI usarlo:**

- Para variables **ordinales reales**, donde el orden existe (`bajo < medio <
  alto`, `First < Second < Third`).
- Para **modelos basados en arboles** (Random Forest, gradient boosting), que
  parten por umbrales y no asumen distancias: el orden arbitrario les molesta poco.

**Cuando NO:** variables **nominales** (sin orden: puerto, color, pais) con modelos
lineales / de distancia / redes. Ahi usa **one-hot** (seccion 3b).

In [ ]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

work_enc = df.dropna(subset=["embarked", "class"]).copy()

# LabelEncoder: pensado para el OBJETIVO (1 columna). Asigna 0,1,2,... por orden alfabetico.
le = LabelEncoder()
embarked_le = le.fit_transform(work_enc["embarked"])
print("LabelEncoder embarked:", dict(zip(le.classes_, range(len(le.classes_)))))

# OrdinalEncoder: para FEATURES. Aqui SI hay orden real en 'class' -> lo imponemos a proposito.
order = [["First", "Second", "Third"]]
oe = OrdinalEncoder(categories=order)
class_oe = oe.fit_transform(work_enc[["class"]])
print("OrdinalEncoder class (orden REAL First<Second<Third):",
      dict(zip(order[0], range(3))))
print("Primeras filas class -> codigo:",
      list(zip(work_enc["class"].head(5), class_oe[:5].ravel().astype(int))))

### 3b. One-hot encoding

**Intuicion primero.** Imagina la columna `embarked` con tres puertos: S, C, Q.
Si los codificas como 0, 1, 2 le estas mintiendo al modelo: le dices que Q (2)
esta "el doble de lejos" de S (0) que C (1), y que hay un orden. Pero entre
puertos no hay orden ni distancia natural. **One-hot** resuelve esto dandole a
cada categoria su propia columna indicadora (0/1), de modo que todas las
categorias quedan a la misma distancia entre si.

**La formula, en compacto.** Una variable categorica con $K$ niveles se mapea a
un vector con un solo 1:

$$\text{onehot}(c) = e_c \in \{0,1\}^{K}, \qquad (e_c)_j = 1 \text{ si } j=c, \text{ si no } 0.$$

Exactamente una componente vale 1, asi que los vectores son ortogonales y
equidistantes: no hay orden espurio.

**Cuidado con la cardinalidad.** El encoding agrega $K$ columnas por feature (o
$K-1$ si *eliminas una* para evitar la trampa de las variables dummy /
colinealidad). Para features de alta cardinalidad (IDs de usuario, codigos
postales) $K$ explota y produce matrices enormes y **dispersas** (casi todo
ceros). Esa es justo la motivacion del *hashing trick* (seccion 2) y de los
*embeddings* (seccion 7).

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ["sex", "embarked"]
work = df.dropna(subset=cat_cols).copy()

# Version comoda con pandas (drop_first evita la trampa de las dummy).
dummies = pd.get_dummies(work[cat_cols], drop_first=False)
print("Columnas de pd.get_dummies:", list(dummies.columns))
dummies.head()

In [ ]:
# Version sklearn: el camino de produccion (ajusta en train, transforma en test).
enc = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X = enc.fit_transform(work[cat_cols])
print("Categorias:", enc.categories_)
print("Forma codificada:", X.shape, "| dtype:", X.dtype, "| guardado como matriz dispersa")
print(f"Densidad: {X.nnz / (X.shape[0]*X.shape[1]):.3f} "
      f"(fraccion de entradas distintas de cero)")
X[:5].toarray()

### 3c. Feature hashing (el "hashing trick")

**Intuicion primero.** One-hot tiene dos problemas: necesita *conocer todas las
categorias de antemano* y crece con la cardinalidad. El hashing es como tener un
archivero con un numero **fijo** de cajones: tomas el nombre de cada categoria,
le aplicas una funcion hash y eso te dice en que cajon va. No importa cuantas
categorias aparezcan: el numero de cajones $m$ **lo eliges tu**.

**La formula, en compacto.** Cada feature (nombre=valor) se manda a uno de $m$
buckets mediante un hash $h$:

$$\phi(x)_j = \sum_{i\,:\,h(i)=j} \xi(i)\, x_i, \qquad j = 0,\dots,m-1,$$

donde $\xi(i)\in\{-1,+1\}$ es un hash de signo que mantiene el mapeo
(aproximadamente) insesgado.

**Colisiones.** Como $m$ es fijo, dos categorias distintas pueden caer en el
mismo cajon (una colision). Con $d$ features activas y $m$ buckets, la
probabilidad de que un par colisione es $\approx 1/m$; elegir $m \gg d$ las hace
raras. El hash de signo $\xi$ hace que las contribuciones que colisionan se
cancelen en promedio en vez de sumarse siempre.

**El trade-off.** El hashing es *sin estado*: no hay vocabulario que guardar,
maneja categorias nuevas gratis y acota la memoria a $m$. El costo es una pequena
perdida de precision (por colisiones) y de interpretabilidad (no puedes mapear un
bucket de vuelta a su categoria).

In [ ]:
from sklearn.feature_extraction import FeatureHasher

# Hashea una columna categorica a un numero FIJO y pequeno de buckets.
n_buckets = 8
hasher = FeatureHasher(n_features=n_buckets, input_type="string")

tokens = [[f"embark={v}"] for v in work["embarked"].astype(str)]
H = hasher.transform(tokens)
print(f"{work['embarked'].nunique()} categorias -> {n_buckets} buckets fijos")
print("Forma hasheada:", H.shape)
pd.DataFrame(H.toarray()[:6],
             index=work["embarked"].head(6).values).round(0)

In [ ]:
# Demostramos colisiones: achicamos el espacio de buckets por debajo de la cardinalidad.
def bucket_of(value, m):
    h = FeatureHasher(n_features=m, input_type="string")
    row = h.transform([[f"town={value}"]]).toarray()[0]
    return int(np.flatnonzero(row)[0]) if row.any() else None

towns = work["embark_town"].dropna().unique()
for m in (16, 2):
    mapping = {t: bucket_of(t, m) for t in towns}
    n_collide = len(mapping) - len(set(mapping.values()))
    print(f"m={m:>2} buckets -> {mapping}  | colisiones: {n_collide}")

## 4. Transformaciones

Cuando una variable continua esta muy **sesgada** o tiene **colas pesadas** (como
`fare`: muchas tarifas bajas y unas pocas altisimas), conviene *transformarla*
antes de modelar. El objetivo es **estabilizar la varianza** y acercar la
distribucion a una forma mas simetrica / normal, lo que ayuda a los modelos
lineales y de distancia.

### 4a. Transformacion logaritmica

**Intuicion primero.** El logaritmo **comprime las colas**: aplasta los valores
grandes y separa los pequenos. Es ideal para datos **positivos y sesgados a la
derecha** (precios, ingresos, conteos). Como $\log(0)$ no existe, se usa
$\log(1+x)$ (`np.log1p`), que ademas maneja bien los ceros.

$$x' = \log(1 + x).$$

### 4b. Box-Cox y Yeo-Johnson (PowerTransformer)

**Intuicion primero.** El log es un caso particular de una familia mas general de
transformaciones de potencia que *buscan automaticamente* el exponente que mejor
normaliza los datos.

- **Box-Cox** requiere $x > 0$ (estrictamente positivos):

$$x^{(\lambda)} = \begin{cases} \dfrac{x^{\lambda}-1}{\lambda} & \lambda \neq 0 \\[4pt] \ln x & \lambda = 0 \end{cases}$$

- **Yeo-Johnson** generaliza Box-Cox para admitir **ceros y negativos**, asi que es
  mas flexible cuando los datos no son estrictamente positivos.

`sklearn.preprocessing.PowerTransformer` estima $\lambda$ por maxima verosimilitud
para **acercar la distribucion a una normal**. Usamos `PowerTransformer` (en vez de
`scipy` directamente) para no agregar dependencias.

In [ ]:
from sklearn.preprocessing import PowerTransformer

fare_raw = df["fare"].dropna().to_numpy(dtype=float)
fare_pos = fare_raw[fare_raw > 0].reshape(-1, 1)   # Box-Cox exige x > 0

def skew(a):
    a = a.ravel(); m = a.mean(); s = a.std()
    return float(((a - m) ** 3).mean() / s ** 3) if s > 0 else 0.0

fare_log = np.log1p(fare_pos)
fare_bc = PowerTransformer(method="box-cox").fit_transform(fare_pos)
fare_yj = PowerTransformer(method="yeo-johnson").fit_transform(fare_pos)

print(f"skew original     : {skew(fare_pos):+.3f}")
print(f"skew log1p        : {skew(fare_log):+.3f}")
print(f"skew Box-Cox      : {skew(fare_bc):+.3f}")
print(f"skew Yeo-Johnson  : {skew(fare_yj):+.3f}  (mas cerca de 0 = mas simetrico)")

In [ ]:
# Antes / despues: la transformacion reduce el sesgo y estabiliza la escala.
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].hist(fare_pos, bins=40, color="#e45756"); ax[0].set_title(f"fare original\nskew={skew(fare_pos):+.2f}")
ax[1].hist(fare_log, bins=40, color="#4c78a8"); ax[1].set_title(f"log1p(fare)\nskew={skew(fare_log):+.2f}")
ax[2].hist(fare_bc, bins=40, color="#54a24b"); ax[2].set_title(f"Box-Cox(fare)\nskew={skew(fare_bc):+.2f}")
for a in ax: a.set_xlabel("valor"); a.set_ylabel("conteo")
plt.tight_layout(); plt.show()

### 4c. Bucketing / binning (discretizacion)

**Intuicion primero.** A veces lo que importa no es el valor exacto sino el
*tramo*: para sobrevivir en el Titanic quiza no importa si tienes 4 o 6 anos,
sino si eres "nino" o "adulto". El binning convierte una variable continua en
cajones ordenados, lo que ayuda a modelos no lineales, domestica outliers y
captura efectos de umbral.

Dos estrategias comunes:

- **Uniforme (igual ancho):** parte el rango $[\min, \max]$ en $k$ intervalos del
  mismo ancho $w = \frac{\max-\min}{k}$. Simple, pero sensible a sesgo/outliers
  (algunos cajones quedan casi vacios).
- **Por cuantiles (igual frecuencia):** pone los bordes en los cuantiles
  $\tfrac{i}{k}$ para que cada cajon tenga ~$n/k$ muestras. Robusto al sesgo; los
  anchos varian.

Columnas de dinero/edad sesgadas (como `fare`) suelen quedar mejor por
**cuantiles**.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

age = df["age"].dropna().to_frame()

# pandas: uniforme (cut) vs cuantiles (qcut)
uniform = pd.cut(age["age"], bins=4)
quantile = pd.qcut(age["age"], q=4)
print("Conteos por bin uniforme (igual ancho):")
print(uniform.value_counts().sort_index())
print("\nConteos por bin de cuantiles (igual frecuencia):")
print(quantile.value_counts().sort_index())

In [ ]:
# sklearn KBinsDiscretizer: cajones codificados como ordinales, listos para un modelo.
kb_uniform = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="uniform")
kb_quant   = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")

age_u = kb_uniform.fit_transform(age).ravel()
age_q = kb_quant.fit_transform(age).ravel()
print("Bordes uniformes:  ", np.round(kb_uniform.bin_edges_[0], 1))
print("Bordes por cuantil:", np.round(kb_quant.bin_edges_[0], 1))

fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
ax[0].hist(age_u, bins=np.arange(-0.5, 4.5), rwidth=0.8); ax[0].set_title("Uniforme")
ax[1].hist(age_q, bins=np.arange(-0.5, 4.5), rwidth=0.8); ax[1].set_title("Cuantiles")
for a in ax: a.set_xlabel("bin"); a.set_xticks(range(4))
ax[0].set_ylabel("conteo"); plt.tight_layout(); plt.show()

## 5. Escalado

Muchos algoritmos miden *distancias* o calculan *gradientes*, asi que la **escala**
de cada feature importa. Aqui vemos cuatro formas de poner las features en una
escala comparable: **estandarizacion (z-score)**, **escalado robusto**, **min-max**
y **normalizacion L2**.

### 5a. Estandarizacion (z-score)

**Intuicion primero.** Muchos algoritmos (regresion lineal/logistica con
regularizacion, SVMs, k-NN, k-means, PCA, redes neuronales) miden *distancias* o
calculan *gradientes*. Si una feature esta en metros y otra en milimetros, la de
milimetros domina solo por estar en una unidad mas grande, no porque importe
mas. La estandarizacion pone a todas las features en la misma escala: media 0 y
varianza 1.

**La formula, en compacto.**

$$z = \frac{x - \mu}{\sigma}, \qquad \mu = \tfrac{1}{n}\sum_i x_i, \quad \sigma = \sqrt{\tfrac{1}{n}\sum_i (x_i-\mu)^2}.$$

Despues de esto cada feature tiene $\mathbb{E}[z]=0$ y $\operatorname{Var}[z]=1$,
asi que ninguna domina una distancia o un gradiente solo por sus unidades.

**Ajusta solo en train.** Calcula $\mu,\sigma$ en el set de entrenamiento y
reutilizalos en validacion/test para evitar fugas de informacion (*leakage*).

In [ ]:
from sklearn.preprocessing import StandardScaler

num_cols = ["age", "fare"]
num = df[num_cols].dropna()

scaler = StandardScaler().fit(num)
z = scaler.transform(num)
print("Medias aprendidas:", scaler.mean_.round(3))
print("Desv. estandar   :", np.sqrt(scaler.var_).round(3))
print("Tras escalar -> media:", z.mean(axis=0).round(3),
      " std:", z.std(axis=0).round(3))

# chequeo manual de la formula del z-score para el primer valor de 'age'
x0 = num["age"].iloc[0]
print(f"\nz manual para age={x0}: "
      f"{(x0 - scaler.mean_[0]) / np.sqrt(scaler.var_[0]):.4f}  "
      f"== sklearn: {z[0,0]:.4f}")

### 5b. Escalado robusto (RobustScaler)

**Intuicion primero.** El z-score usa la **media** y la **desviacion estandar**,
y ambas son *muy sensibles a outliers*: un solo valor extremo arrastra la media y
infla la desviacion, comprimiendo a todos los demas puntos. El **escalado
robusto** cambia esos estadisticos por sus versiones resistentes: centra con la
**mediana** y escala por el **rango intercuartil** ($\text{IQR} = Q_3 - Q_1$, la
distancia entre el percentil 25 y el 75). La mediana y el IQR casi no se mueven
aunque haya valores extremos, asi que la transformacion es **robusta a outliers**.

**La formula, en compacto.**

$$x_{\text{rob}} = \frac{x - \text{mediana}(x)}{Q_3 - Q_1}.$$

Util en columnas con colas pesadas u outliers (precios, ingresos, `fare` del
Titanic). Si la distribucion es limpia, RobustScaler y StandardScaler dan
resultados parecidos; la diferencia aparece justo cuando hay valores extremos.

In [ ]:
from sklearn.preprocessing import RobustScaler

# Tomamos 'fare' (sesgada, con colas) e inyectamos un par de outliers extremos
# para hacer visible la diferencia entre StandardScaler y RobustScaler.
fare = df["fare"].dropna().to_numpy(dtype=float)
fare_out = np.concatenate([fare, [600.0, 800.0]])   # tarifas absurdamente altas
fare_out = fare_out.reshape(-1, 1)

z_fare = StandardScaler().fit_transform(fare_out).ravel()
r_fare = RobustScaler().fit_transform(fare_out).ravel()

print(f"mediana={np.median(fare_out):.2f}  "
      f"Q1={np.percentile(fare_out,25):.2f}  Q3={np.percentile(fare_out,75):.2f}  "
      f"IQR={np.percentile(fare_out,75)-np.percentile(fare_out,25):.2f}")
print(f"StandardScaler -> rango [{z_fare.min():.2f}, {z_fare.max():.2f}]  "
      f"std del grueso (sin outliers): {z_fare[:-2].std():.3f}")
print(f"RobustScaler   -> rango [{r_fare.min():.2f}, {r_fare.max():.2f}]  "
      f"std del grueso (sin outliers): {r_fare[:-2].std():.3f}")

In [ ]:
# Comparacion visual: los outliers DISTORSIONAN el z-score (aplastan el grueso de
# los datos contra 0), mientras que el escalado robusto se mantiene estable.
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)

ax[0].hist(z_fare[:-2], bins=30, color="#4c78a8", alpha=0.8, label="datos normales")
ax[0].hist(z_fare[-2:], bins=5, color="#e45756", alpha=0.9, label="outliers")
ax[0].set_title("StandardScaler (media / desv.)\nlos outliers comprimen el resto")
ax[0].set_xlabel("valor escalado"); ax[0].set_ylabel("conteo"); ax[0].legend()

ax[1].hist(r_fare[:-2], bins=30, color="#4c78a8", alpha=0.8, label="datos normales")
ax[1].hist(r_fare[-2:], bins=5, color="#e45756", alpha=0.9, label="outliers")
ax[1].set_title("RobustScaler (mediana / IQR)\nel grueso conserva su dispersion")
ax[1].set_xlabel("valor escalado"); ax[1].legend()

plt.tight_layout(); plt.show()

### 5c. Normalizacion (min-max y L2)

"Normalizacion" es un termino sobrecargado: nombra dos transformaciones
distintas.

**Min-max** lleva cada feature a un rango fijo, normalmente $[0,1]$. Intuicion:
es como un termometro reescalado donde el minimo es 0 y el maximo es 1.

$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}.$$

Util cuando necesitas entradas acotadas (pixeles de imagen, algunas redes) y la
distribucion no tiene colas pesadas. Es **sensible a outliers**: un valor extremo
estira el rango y comprime a todos los demas.

**Normalizacion L2 (vectorial)** reescala cada *muestra* (fila) a longitud 1.
Intuicion: te quedas solo con la *direccion* del vector, no con su magnitud.

$$\mathbf{x}' = \frac{\mathbf{x}}{\lVert \mathbf{x}\rVert_2}, \qquad \lVert \mathbf{x}\rVert_2 = \sqrt{\textstyle\sum_j x_j^2}.$$

Ahora $\lVert \mathbf{x}'\rVert_2 = 1$. Es comun en texto/TF-IDF y donde se usa
similitud coseno. Ojo a la diferencia: StandardScaler/MinMax actuan **por
columna**; Normalizer actua **por fila**.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, Normalizer

mm = MinMaxScaler().fit(num)
num_mm = mm.transform(num)
print("min-max -> min:", num_mm.min(axis=0).round(3),
      " max:", num_mm.max(axis=0).round(3))

# Normaliza L2 cada FILA (muestra) de los vectores de dos features.
l2 = Normalizer(norm="l2")
num_l2 = l2.fit_transform(num.values[:5])
print("\nPrimeras 5 filas tras normalizacion L2:")
print(np.round(num_l2, 3))
print("Normas de fila (deben dar 1):", np.round(np.linalg.norm(num_l2, axis=1), 3))

## 6. Reduccion de dimensionalidad — PCA

**Intuicion primero.** Imagina que fotografias una nube de puntos 3D. Quieres
elegir el angulo de la camara que muestre la *mayor variacion* posible: ese
angulo "ve" mas estructura. PCA hace exactamente eso pero en cualquier numero de
dimensiones: encuentra direcciones (las **componentes principales**) ordenadas
por cuanta varianza capturan, y te quedas con las primeras.

**La formula, en compacto.** Con datos estandarizados $X \in \mathbb{R}^{n\times d}$
se forma la matriz de covarianza

$$\Sigma = \frac{1}{n-1} X^\top X,$$

y se hace su descomposicion en autovalores: $\Sigma v_k = \lambda_k v_k$. Los
autovectores $v_k$ son las direcciones (ortonormales); el autovalor $\lambda_k$ es
la varianza a lo largo de $v_k$. La **proporcion de varianza explicada** por la
componente $k$ es

$$\text{EVR}_k = \frac{\lambda_k}{\sum_j \lambda_j}.$$

Proyectando sobre las $r$ primeras componentes, $Z = X V_r$, obtienes una
representacion mas compacta que retiene la mayor varianza posible.
**Estandariza primero**, o las features de mayor varianza (unidades grandes)
dominaran las componentes.

In [ ]:
from sklearn.decomposition import PCA

pca_cols = ["age", "fare", "sibsp", "parch", "pclass"]
Xp = df[pca_cols].dropna()
Xp_std = StandardScaler().fit_transform(Xp)

pca = PCA().fit(Xp_std)
evr = pca.explained_variance_ratio_
print("Varianza explicada por componente:", evr.round(3))
print("Acumulada:", np.cumsum(evr).round(3))

# Chequeo: los autovalores de PCA == autovalores de la matriz de covarianza.
cov = np.cov(Xp_std, rowvar=False)
eigvals = np.sort(np.linalg.eigvalsh(cov))[::-1]
print("\nPCA explained_variance_:", pca.explained_variance_.round(3))
print("Autovalores np.linalg  :", eigvals.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) scree / varianza explicada acumulada
ax[0].bar(range(1, len(evr)+1), evr, alpha=0.6, label="individual")
ax[0].plot(range(1, len(evr)+1), np.cumsum(evr), "o-r", label="acumulada")
ax[0].axhline(0.9, ls="--", c="gray"); ax[0].set_xlabel("componente")
ax[0].set_ylabel("varianza explicada"); ax[0].set_title("Scree plot"); ax[0].legend()

# (b) datos proyectados sobre las dos primeras componentes principales
Z = PCA(n_components=2).fit_transform(Xp_std)
target = df.loc[Xp.index, "survived"]
sc = ax[1].scatter(Z[:, 0], Z[:, 1], c=target, cmap="coolwarm", s=12, alpha=0.6)
ax[1].set_xlabel("PC1"); ax[1].set_ylabel("PC2")
ax[1].set_title("Titanic proyectado sobre PC1/PC2"); plt.colorbar(sc, ax=ax[1], label="survived")
plt.tight_layout(); plt.show()

## 7. Embeddings — vectores densos aprendidos

**Intuicion primero.** Los vectores one-hot son **dispersos, enormes y
equidistantes**: cada categoria esta a la misma distancia de todas las demas, asi
que no codifican *parecido*. Un **embedding** le asigna a cada categoria un vector
denso *aprendido* $\mathbf{e}_c \in \mathbb{R}^{d}$ con $d \ll K$. La magia: como
ese vector se aprende junto con el modelo, categorias parecidas terminan *cerca*
en el espacio (piensa en "rey" y "reina" quedando cerca en NLP).

**La formula, en compacto.**

$$c \;\longmapsto\; \mathbf{e}_c = E^\top\, \text{onehot}(c), \qquad E \in \mathbb{R}^{K\times d}.$$

La matriz $E$ (la *tabla de embeddings*) se **entrena por descenso de gradiente
junto con el modelo**. Mecanicamente, buscar un embedding es solo *seleccionar
una fila* de $E$: equivale a multiplicar el one-hot por $E$, pero se hace como un
indice $O(1)$ en lugar de una multiplicacion de matrices dispersa.

Los embeddings son la forma estandar de manejar categoricas de alta cardinalidad
(usuarios, productos, palabras) y son la base del NLP moderno y de los sistemas
de recomendacion.

La idea visual de un embedding: cada palabra/categoria se ubica como un punto en
un espacio denso, y **la cercania codifica similitud semantica**. En el diagrama
de abajo (posiciones ilustrativas, no aprendidas) los animales quedan juntos y los
vehiculos quedan juntos, separados entre si.

In [ ]:
# Diagrama conceptual: categorias -> vectores densos en un espacio 2D
# donde cercania = similitud semantica (posiciones ILUSTRATIVAS).
puntos = {
    "perro":  (1.2, 2.6), "gato":  (1.6, 2.2),   # cluster: animales
    "auto":   (4.0, 1.0), "camion": (4.4, 1.4),  # cluster: vehiculos
}
colores = {"perro": "#1f77b4", "gato": "#1f77b4",
           "auto": "#d62728", "camion": "#d62728"}

fig, ax = plt.subplots(figsize=(7, 5))
for palabra, (x, y) in puntos.items():
    ax.scatter(x, y, s=180, color=colores[palabra], zorder=3, edgecolor="white")
    ax.annotate(palabra, (x, y), textcoords="offset points", xytext=(10, 6),
                fontsize=12, fontweight="bold")

# Elipses sutiles para resaltar los dos clusters semanticos.
from matplotlib.patches import Ellipse
ax.add_patch(Ellipse((1.4, 2.4), 1.6, 1.3, fc="#1f77b4", alpha=0.12, zorder=1))
ax.add_patch(Ellipse((4.2, 1.2), 1.6, 1.3, fc="#d62728", alpha=0.12, zorder=1))
ax.text(1.4, 3.25, "animales", ha="center", color="#1f77b4", fontsize=11)
ax.text(4.2, 0.4, "vehiculos", ha="center", color="#d62728", fontsize=11)

ax.set_xlabel("dimension latente 1"); ax.set_ylabel("dimension latente 2")
ax.set_title("Embeddings: cercania en el espacio = similitud semantica")
ax.set_xlim(0, 6); ax.set_ylim(-0.2, 3.8)
plt.tight_layout(); plt.show()

Y el contraste estructural: un vector **one-hot** es largo y disperso (casi todo
ceros, un unico 1), mientras que un **embedding** es corto y denso (pocos numeros
reales que concentran la informacion).

In [ ]:
# One-hot (ancho y disperso) vs embedding (corto y denso) para la misma categoria.
onehot = np.zeros(12); onehot[3] = 1            # 1 en la posicion de la categoria
embed = np.array([0.21, -0.84, 0.57])           # 3 numeros densos aprendidos

fig, ax = plt.subplots(2, 1, figsize=(11, 3.2),
                       gridspec_kw={"height_ratios": [1, 1]})

ax[0].imshow(onehot.reshape(1, -1), cmap="Blues", aspect="auto", vmin=0, vmax=1)
for j, v in enumerate(onehot):
    ax[0].text(j, 0, str(int(v)), ha="center", va="center",
               color="white" if v else "#333", fontsize=10)
ax[0].set_title("One-hot: 12-d, disperso (un solo 1, el resto 0)")
ax[0].set_yticks([]); ax[0].set_xticks(range(12))

ax[1].imshow(embed.reshape(1, -1), cmap="coolwarm", aspect="auto", vmin=-1, vmax=1)
for j, v in enumerate(embed):
    ax[1].text(j, 0, f"{v:.2f}", ha="center", va="center", fontsize=11)
ax[1].set_title("Embedding: 3-d, denso (numeros reales con significado aprendido)")
ax[1].set_yticks([]); ax[1].set_xticks(range(3))

plt.tight_layout(); plt.show()

In [ ]:
# Ejemplo minimo de nn.Embedding (PyTorch) para la categorica 'embark_town'.
# (Cae a una ilustracion con NumPy si torch no esta disponible.)
sex_cat = df["embark_town"].dropna().astype("category")
vocab = list(sex_cat.cat.categories)
idx = sex_cat.cat.codes.to_numpy()
print("Vocabulario:", vocab)

try:
    import torch
    import torch.nn as nn
    torch.manual_seed(0)

    embed_dim = 3
    emb = nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim)
    sample = torch.tensor(idx[:5])
    vectors = emb(sample)
    print(f"\nnn.Embedding: {len(vocab)} categorias -> vectores densos de {embed_dim}-d")
    print("Tabla de embeddings E (una fila por categoria):")
    print(emb.weight.detach().numpy().round(3))
    print("\nBusqueda para las primeras 5 filas (indices", idx[:5], "):")
    print(vectors.detach().numpy().round(3))
    _torch_ok = True
except Exception as e:
    print("torch no disponible, ilustracion con NumPy:", e)
    rng = np.random.default_rng(0)
    E = rng.normal(0, 1, size=(len(vocab), 3))
    print("Tabla de embeddings E:\n", E.round(3))
    print("Busqueda (filas", idx[:5], "):\n", E[idx[:5]].round(3))
    _torch_ok = False

In [ ]:
# Por que los embeddings ganan a one-hot en alta cardinalidad: conteo de parametros/memoria.
K = df["embark_town"].nunique()
for K_demo in [K, 1000, 1_000_000]:
    onehot_dim = K_demo               # ancho de un vector one-hot
    emb_params = K_demo * 8           # una tabla de embeddings de 8-d
    print(f"cardinalidad {K_demo:>9,}: ancho one-hot = {onehot_dim:>9,} (disperso), "
          f"tabla embedding (d=8) = {emb_params:>10,} params (denso, reutilizable)")

## Resumen — tabla de referencia rapida

| Tecnica | Que hace | Formula | Usar cuando | Cuidado con |
|---|---|---|---|---|
| **Seleccion (filter/wrapper/embedded)** | elige un subconjunto de columnas | MI, chi², RFE, L1 | reducir ruido/redundancia conservando interpretabilidad | filtros ignoran interacciones; wrappers son caros |
| **Imputacion** | rellena faltantes | media/mediana/moda, KNN, indicador | el modelo no acepta NaN | el supuesto sesga; ajusta solo en train |
| **Label / Ordinal** | categoria → entero | $c \mapsto k$ | ordinales reales o modelos de arbol | impone orden falso en nominales |
| **One-hot** | categoria → vector indicador | $e_c\in\{0,1\}^K$ | categoricas de baja cardinalidad | explota / dispersa con $K$ alto; elimina una para evitar colinealidad |
| **Feature hashing** | categoria → buckets fijos via hash | $\phi(x)_j=\sum_{h(i)=j}\xi(i)x_i$ | cardinalidad alta/desconocida, streaming | colisiones; no interpretable |
| **Log / log1p** | comprime colas | $\log(1+x)$ | datos positivos sesgados (precios, conteos) | requiere $x>-1$; log puro no admite 0 |
| **Box-Cox / Yeo-Johnson** | potencia que normaliza | $\frac{x^\lambda-1}{\lambda}$ (BC) | estabilizar varianza, acercar a normal | Box-Cox exige $x>0$; YJ admite 0 y negativos |
| **Binning** | continua → cajones ordinales | ancho uniforme vs bordes por cuantil | umbrales, domesticar sesgo/outliers | el numero de bins es hiperparametro; pierde info |
| **Estandarizacion** | media 0, varianza 1 | $z=\frac{x-\mu}{\sigma}$ | SVM, k-NN, PCA, lineal+reg, NN | ajustar solo en train; sensible a outliers |
| **Escalado robusto** | centra con mediana, escala por IQR | $\frac{x-\text{mediana}}{Q_3-Q_1}$ | features con outliers / colas pesadas (precios, ingresos) | no acota a un rango fijo |
| **Min-max** | escalar a $[0,1]$ | $\frac{x-x_{\min}}{x_{\max}-x_{\min}}$ | entradas acotadas (pixeles) | los outliers estiran el rango |
| **Normalizar L2** | *filas* de longitud 1 | $\mathbf{x}/\lVert\mathbf{x}\rVert_2$ | similitud coseno, TF-IDF | es por muestra, no por feature |
| **PCA** | proyeccion que maximiza varianza | autovec. de $\Sigma=\frac{1}{n-1}X^\top X$ | decorrelacionar, comprimir, visualizar | estandarizar primero; pierde interpretabilidad |
| **Embeddings** | vectores densos aprendidos | $\mathbf{e}_c=E^\top\text{onehot}(c)$ | alta cardinalidad, importa el parecido | requiere datos de entrenamiento; es parte del modelo |

**Reglas de oro**

1. **Ajusta las transformaciones solo en el split de entrenamiento** y luego
   aplicalas a val/test (sin leakage). En produccion, persiste el transformador
   ajustado o, mejor aun, usa un **feature store** (Notebook 3).
2. Empareja la transformacion con el *modelo*: a los ensembles de arboles casi no
   les importa el escalado; a los modelos basados en distancia/gradiente, mucho.
3. Prefiere pipelines (`sklearn.pipeline.Pipeline`, `ColumnTransformer`) para que
   los mismos pasos corran identicos en entrenamiento y en serving.